In [ ]:
import os\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nPROJECT_ROOT = Path.cwd().resolve()\nif not (PROJECT_ROOT / 'src').exists():\n    PROJECT_ROOT = PROJECT_ROOT.parent\nsys.path.insert(0, str(PROJECT_ROOT))\nos.chdir(PROJECT_ROOT)\n\nPath('data').mkdir(exist_ok=True)\n\nN_TRAIN = 50\nN_VAL = 10\nN_TEST = 20\nH = W = 28\nN_CLASSES = 3\nrng = np.random.default_rng(42)\n\nnp.savez(\n    'data/smokemnist.npz',\n    train_images=rng.integers(0, 256, size=(N_TRAIN, H, W, 3), dtype=np.uint8),\n    train_labels=rng.integers(0, N_CLASSES, size=(N_TRAIN, 1), dtype=np.int64),\n    val_images=rng.integers(0, 256, size=(N_VAL, H, W, 3), dtype=np.uint8),\n    val_labels=rng.integers(0, N_CLASSES, size=(N_VAL, 1), dtype=np.int64),\n    test_images=rng.integers(0, 256, size=(N_TEST, H, W, 3), dtype=np.uint8),\n    test_labels=rng.integers(0, N_CLASSES, size=(N_TEST, 1), dtype=np.int64),\n)\n

In [ ]:
import torch\nfrom torch.utils.data import DataLoader\n\nfrom src.dataset import MedMNISTDataset\nfrom src.transforms import get_train_transform, get_val_transform\n\ndata = np.load('data/smokemnist.npz')\ntrain_transform = get_train_transform(224, 'light', True, 'smokemnist')\nval_transform = get_val_transform(224, True)\n\ntrain_dataset = MedMNISTDataset(data['train_images'], data['train_labels'], transform=train_transform)\nval_dataset = MedMNISTDataset(data['val_images'], data['val_labels'], transform=val_transform)\ntest_dataset = MedMNISTDataset(data['test_images'], data['test_labels'], transform=val_transform)\n\ntrain_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)\nval_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)\ntest_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)\n\nimages, labels = next(iter(train_loader))\nassert tuple(images.shape[1:]) == (3, 224, 224)\nassert labels.dtype == torch.int64\nassert int(labels.min()) >= 0 and int(labels.max()) < N_CLASSES\n

In [ ]:
import timm\n\nfrom src.models import freeze_backbone, unfreeze_all\n\nmodel = timm.create_model('resnet18', pretrained=False, num_classes=N_CLASSES, in_chans=3)\nlogits = model(images)\nassert tuple(logits.shape) == (images.shape[0], N_CLASSES)\n\nfreeze_backbone(model)\nassert any(param.requires_grad for name, param in model.named_parameters() if 'fc' in name)\nassert not any(param.requires_grad for name, param in model.named_parameters() if 'fc' not in name)\n\nunfreeze_all(model)\nassert all(param.requires_grad for param in model.parameters())\n

In [ ]:
from torch import nn\n\nfrom src.train import train_one_epoch, validate\n\ndevice = torch.device('cpu')\nmodel.to(device)\ncriterion = nn.CrossEntropyLoss()\noptimizer = torch.optim.Adam(model.parameters(), lr=1e-3)\n\nfor epoch in range(2):\n    train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, device)\n    val_loss, val_f1, per_class_f1 = validate(model, val_loader, criterion, device)\n    print(f'epoch={epoch + 1} train_loss={train_loss:.4f} train_f1={train_f1:.4f} val_loss={val_loss:.4f} val_f1={val_f1:.4f}')\n

In [ ]:
from pathlib import Path\n\nimport src.ensemble as ensemble_module\nfrom src.ensemble import load_model_from_checkpoint\n\nckpt_dir = Path('checkpoints/smoke')\nckpt_dir.mkdir(parents=True, exist_ok=True)\nckpt_path = ckpt_dir / 'test_ckpt.pth'\ntorch.save({'model_state_dict': model.state_dict()}, ckpt_path)\n\nensemble_module.build_model = lambda model_name, n_classes: timm.create_model(\n    'resnet18', pretrained=False, num_classes=n_classes, in_chans=3\n)\nloaded_model = load_model_from_checkpoint(ckpt_path, 'resnet18', N_CLASSES, device)\nwith torch.no_grad():\n    loaded_logits = loaded_model(images)\nassert loaded_logits.shape[1] == N_CLASSES\n

In [ ]:
import pandas as pd\n\nfrom configs.dataset_configs import DATASET_CONFIGS\nfrom src.submission import build_submission_csv\n\npredictions = {}\nfor dataset_name, config in DATASET_CONFIGS.items():\n    n_test = 4\n    if config['is_rgb']:\n        test_images = rng.integers(0, 256, size=(n_test, 28, 28, 3), dtype=np.uint8)\n    else:\n        test_images = rng.integers(0, 256, size=(n_test, 28, 28), dtype=np.uint8)\n\n    np.savez(\n        f'data/{dataset_name}.npz',\n        train_images=test_images,\n        train_labels=np.zeros((n_test, 1), dtype=np.int64),\n        val_images=test_images,\n        val_labels=np.zeros((n_test, 1), dtype=np.int64),\n        test_images=test_images,\n    )\n    predictions[dataset_name] = rng.integers(0, config['n_classes'], size=n_test, dtype=np.int64)\n\noutput_path = Path('submissions/smoke_submission.csv')\nsubmission = build_submission_csv(predictions, output_path)\nassert list(submission.columns) == ['id', 'id_image_in_task', 'task_name', 'label']\nassert submission['id'].is_monotonic_increasing\nassert set(submission['task_name']) == set(DATASET_CONFIGS.keys())\nfor _, group in submission.groupby('task_name'):\n    assert int(group['id_image_in_task'].iloc[0]) == 0\nassert len(pd.read_csv(output_path)) == len(submission)\noutput_path.unlink()\n

In [ ]:
print('ALL SMOKE TESTS PASSED')\n